<a href="https://colab.research.google.com/github/anastasia-kotselivska/ITA/blob/main/%D1%81olloquium_experiment/exp1_lab4_ocr_degradation_var6.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [12]:
!pip install -q easyocr opencv-python

import cv2
import easyocr
import numpy as np

VARIANT = "Variant 6 (receipt.jpg)"
IMAGE_PATH = "receipt.jpg"


GT_TEXT = (
    "М ІВАНО-ФРАНКІВСЬК ВУЛ: КОБИЛЯНСЬКОЇ, 56 ПРИВАТБАНК "
    "1049932 Eskaro Стрічка малярна 351.14 Знижка 59.79 "
    "654422 GRUVER Герметик акриловий 258.02 "
    "634842 GRUVER Клей рідкий 732.35 "
    "589822 Kirchland PrimeTex 10л Грунт 591.70 "
    "Сума 1933.21 13.10.2026"
)

def edit_distance(seq_a, seq_b) -> int:
  n, m = len(seq_a), len(seq_b)
  dp = [[0] * (m + 1) for _ in range(n + 1)]
  for i in range(n + 1):
    dp[i][0] = i
  for j in range(m + 1):
    dp[0][j] = j
  for i in range(1, n + 1):
    for j in range(1, m + 1):
      cost = 0 if seq_a[i - 1] == seq_b[j - 1] else 1
      dp[i][j] = min(
          dp[i - 1][j] + 1,  # Deletion (D)
          dp[i][j - 1] + 1,  # Insertion (I)
          dp[i - 1][j - 1] + cost,  # Substitution (S)
      )
  return dp[n][m]


def compute_cer_wer(ref: str, hyp: str):
  ref_chars = list(ref.replace(" ", ""))
  hyp_chars = list(hyp.replace(" ", ""))
  cer = (
      edit_distance(ref_chars, hyp_chars) / len(ref_chars) if ref_chars else 1.0
  )

  ref_words = ref.split()
  hyp_words = hyp.split()
  wer = (
      edit_distance(ref_words, hyp_words) / len(ref_words) if ref_words else 1.0
  )
  return cer, wer


def run_ocr_filtered(img_gray, conf_min: float):
  res = reader.readtext(img_gray)
  kept = [
      (txt.strip(), float(conf))
      for (_, txt, conf) in res
      if float(conf) > conf_min and txt.strip()
  ]
  dropped = len(res) - len(kept)
  avg_conf = float(np.mean([c for _, c in kept])) if kept else 0.0
  text_out = " ".join([t for t, _ in kept])
  return text_out, len(kept), dropped, avg_conf


img = cv2.imread(IMAGE_PATH)
gray_img = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
reader = easyocr.Reader(["uk", "en"], gpu=False)

# --- РЕЖИМ "ДО" ---
beta_before, conf_before = 0, 0.30
img_before = cv2.convertScaleAbs(gray_img, alpha=1.0, beta=beta_before)
text_before, cnt_before, drop_before, avg_conf_before = run_ocr_filtered(
    img_before, conf_before
)
cer_before, wer_before = compute_cer_wer(GT_TEXT, text_before)

print(f"=== ДО | {VARIANT} ===")
print(
    f"beta={beta_before} | conf>{conf_before:.2f} | Kept blocks={cnt_before} |"
    f" Dropped={drop_before} | Avg Conf={avg_conf_before:.4f}"
)
print(
    f"CER={cer_before:.4f} ({cer_before*100:.2f}%) | WER={wer_before:.4f}"
    f" ({wer_before*100:.2f}%)\n"
)

# --- РЕЖИМ "ПІСЛЯ" ---
beta_after, conf_after = 130, 0.75
img_after = cv2.convertScaleAbs(gray_img, alpha=1.0, beta=beta_after)
text_after, cnt_after, drop_after, avg_conf_after = run_ocr_filtered(
    img_after, conf_after
)
cer_after, wer_after = compute_cer_wer(GT_TEXT, text_after)

print(f"=== ПІСЛЯ | {VARIANT} ===")
print(
    f"beta={beta_after} | conf>{conf_after:.2f} | Kept blocks={cnt_after} |"
    f" Dropped={drop_after} | Avg Conf={avg_conf_after:.4f}"
)
print(
    f"CER={cer_after:.4f} ({cer_after*100:.2f}%) | WER={wer_after:.4f}"
    f" ({wer_after*100:.2f}%)\n"
)

# --- ПРИКЛАД ДЛЯ ОДНОГО СЛОВА З ЧЕКА ---
w_ref, w_hyp = "Eskaro", "Eskar"
cer_w, wer_w = compute_cer_wer(w_ref, w_hyp)
print(f"Одне слово з чека: '{w_ref}' -> '{w_hyp}'")
print(
    f"CER = 1/6 = {cer_w:.4f} ({cer_w*100:.2f}%) | WER = 1/1 = {wer_w:.4f}"
    f" ({wer_w*100:.2f}%)"
)

=== ДО | Variant 6 (receipt.jpg) ===
beta=0 | conf>0.30 | Kept blocks=62 | Dropped=5 | Avg Conf=0.7500
CER=2.5213 (252.13%) | WER=3.0312 (303.12%)

=== ПІСЛЯ | Variant 6 (receipt.jpg) ===
beta=130 | conf>0.75 | Kept blocks=3 | Dropped=62 | Avg Conf=0.9065
CER=0.9858 (98.58%) | WER=1.0000 (100.00%)

Одне слово з чека: 'Eskaro' -> 'Eskar'
CER = 1/6 = 0.1667 (16.67%) | WER = 1/1 = 1.0000 (100.00%)
